# Chapter 14 — Problem Set 1: Solutions

Reference solutions for `problem_set_1.ipynb`. All cells run offline on CPU.

---
*Generated by Berta AI*

In [ ]:
import os, sys, json
from pathlib import Path
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from dataset_utils import format_instruction, load_jsonl, tokenize_budget, build_loss_mask
from training_utils import simple_sft_loop
np.random.seed(42)

## Solution 1 — Format an Instruction Dataset

In [ ]:
rows = [
    {'instruction': 'Translate to Spanish', 'input': 'good morning', 'output': 'buenos días'},
    {'instruction': 'Sum the integers',     'input': '4 and 5',      'output': '9'},
]
for r in rows:
    f = format_instruction(**r)
    print('PROMPT:'); print(f['prompt'])
    print('RESPONSE:'); print(f['response'])
    assert f['prompt'].endswith('### Response:\n')
    print('-' * 40)

## Solution 2 — Token Budget

In [ ]:
raw = load_jsonl(Path('..') / '..' / 'datasets' / 'instructions.jsonl')
formatted = [format_instruction(r['instruction'], r.get('input', ''), r.get('output', '')) for r in raw]
for limit in (32, 64, 128):
    s = tokenize_budget(formatted, max_seq_len=limit)
    print(f'max_seq_len={limit:3d}  fit_fraction={s["fit_fraction"]:.2f}  mean={s["mean"]:.1f}  p95={s["p95"]:.1f}  max={s["max"]}')
lens = [len(f['text'].split()) for f in formatted]
plt.hist(lens, bins=12); plt.xlabel('whitespace tokens'); plt.ylabel('count'); plt.title('Token length histogram'); plt.show()

## Solution 3 — Loss Masking

In [ ]:
def make_label_ids(prompt_ids, response_ids, ignore_index=-100):
    return [ignore_index] * len(prompt_ids) + list(response_ids)

prompt_ids = [101, 102, 103, 104]
response_ids = [201, 202, 203]
labels = make_label_ids(prompt_ids, response_ids)
assert len(labels) == len(prompt_ids) + len(response_ids)
assert all(x == -100 for x in labels[:len(prompt_ids)])
assert labels[len(prompt_ids):] == response_ids
print('labels =', labels)

## Solution 4 — Hyperparameter Choices

In [ ]:
hparams = {
    'learning_rate': 2e-4,    # standard for LoRA on a 7B base; full FT would be 1e-5 to 5e-5
    'batch_size': 8,          # fits a single 24GB GPU; use grad accumulation if smaller
    'epochs': 3,              # 1k examples; 3 epochs is a good first try (watch for overfitting)
    'warmup_ratio': 0.03,     # short warmup helps stability
    'weight_decay': 0.01,     # mild regularization
    'grad_clip': 1.0,         # standard for LM training
    'lr_scheduler': 'cosine', # cosine decay after warmup
    'eval_strategy': 'epoch', # check val each epoch and early-stop if regressing
}
print(json.dumps(hparams, indent=2))

## Solution 5 — Fine-tune vs RAG

In [ ]:
answers = {
    1: 'RAG — corpus updates constantly, you cannot retrain every change.',
    2: 'Fine-tune — fixed tone and JSON schema; SFT enforces format better than prompts.',
    3: 'Fine-tune — 5k paired examples is plenty; the task is well-defined.',
    4: 'RAG — current events change daily; retrieval keeps facts fresh.',
    5: 'Fine-tune carefully (LoRA) + held-out evals; 200 examples is small but the task is narrow and accuracy matters.',
}
for k, v in answers.items():
    print(f'{k}. {v}')

## Solution 6 — Tiny SFT Loop with Overfitting and Early Stopping

In [ ]:
rng = np.random.default_rng(0)
n, d, k = 60, 6, 3
X = rng.normal(size=(n, d))
y = (X[:, :k].argmax(axis=1)).astype(int)
X = np.hstack([X, rng.normal(scale=0.5, size=(n, 4))])  # noise dims to encourage overfitting
mask = np.ones(n, dtype=int)

long = simple_sft_loop(X, y, mask, n_classes=k, epochs=80, batch_size=8, lr=0.5,
                       weight_decay=0.0, val_split=0.3, seed=1)
early = simple_sft_loop(X, y, mask, n_classes=k, epochs=80, batch_size=8, lr=0.5,
                        weight_decay=0.0, val_split=0.3, early_stop_patience=2, seed=1)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(long['history']['train_loss'], label='train'); ax[0].plot(long['history']['val_loss'], label='val')
ax[0].set_title('No early stopping'); ax[0].legend()
ax[1].plot(early['history']['train_loss'], label='train'); ax[1].plot(early['history']['val_loss'], label='val')
ax[1].set_title(f'Early stopped at epoch {len(early["history"]["val_loss"])}'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f'Long run epochs: {len(long["history"]["val_loss"])}, early-stop epochs: {len(early["history"]["val_loss"])}')

---
*Generated by Berta AI*